In [4]:
"""
Day 5 experiment: A/B test prompt versions on the expanded test set.
"""


from pathlib import Path
import sys, json
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

import json
from pathlib import Path
from src.retrieval import HybridRetriever
from src.generation import Generator
from src.pipeline import RAGPipeline
from src.evaluation import (
    TestQuestion, LLMJudge, Evaluator, summarize_results, save_results
)

# Build shared infrastructure once
retriever = HybridRetriever(
    chroma_path=str(PROJECT_ROOT / "chroma_db"),
    collection_name="financial_risk_10k",
)
judge = LLMJudge()  # Claude, claude-opus-5-5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading chunks from collection 'financial_risk_10k'...
Building BM25 index over 1401 chunks...
✓ HybridRetriever ready: 1401 chunks


In [ ]:

# Load expanded test set
test_data = json.loads((PROJECT_ROOT / "data" / "evaluation_test_set" / "test_set.json").read_text())
questions = [TestQuestion(**row) for row in test_data]
print(f"Test set: {len(questions)} questions")

# ---- Run A: baseline prompt ----
print("\n=== Run A: v1_baseline ===")
generator_a = Generator(prompt_version="v1_baseline")
pipeline_a = RAGPipeline(retriever=retriever, generator=generator_a, top_k=5)
evaluator_a = Evaluator(pipeline=pipeline_a, judge=judge, delay_seconds=0)
results_a = evaluator_a.evaluate_all(questions)
summary_a = summarize_results(results_a)
save_results(results_a, str(PROJECT_ROOT / "data" / "eval_run_v1_baseline.json"))

# ---- Run B: enforced shape prompt ----
print("\n=== Run B: v2_enforced_shape ===")
generator_b = Generator(prompt_version="v2_enforced_shape")
pipeline_b = RAGPipeline(retriever=retriever, generator=generator_b, top_k=5)
evaluator_b = Evaluator(pipeline=pipeline_b, judge=judge, delay_seconds=0)
results_b = evaluator_b.evaluate_all(questions)
summary_b = summarize_results(results_b)
save_results(results_b, str(PROJECT_ROOT / "data" / "eval_run_v2_enforced_shape.json"))

# ---- Compare ----
print("\n" + "=" * 60)
print("A/B COMPARISON")
print("=" * 60)
print(f"\n{'Metric':<25} {'v1 baseline':<15} {'v2 enforced':<15} {'Δ':<10}")
print("-" * 65)

def fmt(v):
    return f"{v:.3f}" if v is not None else "—"

metrics_to_compare = [
    ("avg_precision_at_5", "retrieval"),
    ("avg_recall_at_5", "retrieval"),
    ("avg_mrr", "retrieval"),
    ("avg_faithfulness", "generation"),
    ("avg_relevance", "generation"),
]

for metric, section in metrics_to_compare:
    va = summary_a[section][metric]
    vb = summary_b[section][metric]
    delta = (vb - va) if (va is not None and vb is not None) else None
    delta_str = f"{delta:+.3f}" if delta is not None else "—"
    print(f"{metric:<25} {fmt(va):<15} {fmt(vb):<15} {delta_str:<10}")

Test set: 18 questions

=== Run A: v1_baseline ===
[1/18] single_01: What cybersecurity risks does JPMorgan disclose in its Risk ...
  ❌ Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
[2/18] single_02: How does Capital One describe its commercial real estate exp...
  P@5=0.60  R@5=1.00  faith=5  rel=3
[3/18] single_03: What climate-related risks does BlackRock identify?...
  ❌ Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
[4/18] single_04: What are PayPal's main risks from third-party service provid...
  ❌ Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', '

KeyboardInterrupt: 

: 

Retrieval doesn't depend on the prompt -- the same chunks come back regardless. If retrieval numbers differ meaningfully, something is wrong. The faithfullness stayed similar, but relevance improved.

# Experiment 001 — Prompt Comparison (Paused)

**Status:** Test set authored (n=18), spot-checked, ready to run.
A/B evaluation blocked on Gemini judge model availability.

## Setup completed
- 18-question golden test set in `data/test_set.json`
- `src/generation.py` supports `prompt_version="v1_baseline"` and `"v2_enforced_shape"`
- Baseline run (n=4, v2_enforced_shape) results in `data/eval_run_baseline.json`

## Resume steps when judge model is available
1. Run notebook `notebooks/run_evaluation.ipynb` twice — once per prompt version
2. Diff with the comparison code from Day 5 plan
3. Document delta in this file

## Why paused
Gemini 2.5 Flash (judge model) returned UNAVAILABLE / rate limit errors
during attempted A/B run. Free tier capacity issue, not a code issue.